### Getting the deputies data

We will work with the LVI (56th) period of the "Cámara de Diputadas y Diputados" (Deputies' Chamber), the dates of this period are from 2022-03-11 to 2026-03-10.

We want to get the deputy ID (dipid), names, gender, date of birth and political militancies.

Since 2026-03-11 starts a new cohort of deputies I will use a specific call to get the information of the deputies, instead of actual period, this will not work while working the code and also not past the 11-03

There's a difference between Periodo legislativo (4 year term of the Chamber), and the Legislatura (1year subdivision in the periodo Legislativo)

In [1]:
import requests
import sqlite3
import pandas as pd
import xml.etree.ElementTree as et
from datetime import datetime


From the database that the Biblioteca Nacional del Congreso (ADD THE LINK) we see that the Periodo legislativo that we want to work with is the officially named LVI (56th), but in the inner database the whole period is numbered with ID=10

[This is where we can find the endpoints from the Cámara del Congreso](https://www.camara.cl/transparencia/datosAbiertos.aspx)

In [2]:
# Simple clean function to handle None values and strip whitespace
def clean(text):
    if text is None:
        return None
    return text.strip()

In [3]:
# this is the endpoint URL that we will work on
url = "https://opendata.camara.cl/camaradiputados/WServices/WSDiputado.asmx/retornarDiputadosXPeriodo"

# calling the API
# the id we are usign now is 10, that's the Period 2022-2026
response = requests.get(url, params={"prmPeriodoID":10}) #add timeout?

# this is the answer we get
deputies_xml = response.text

# save the data so we don't have to download it again
with open(f"diputados_periodo_10.xml", "w", encoding="utf-8") as f:
    f.write(deputies_xml)

In [4]:
deputies_xml

'<?xml version="1.0" encoding="utf-8"?>\r\n<DiputadosPeriodoColeccion xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns="http://opendata.camara.cl/camaradiputados/v1">\r\n  <DiputadoPeriodo>\r\n    <FechaInicio>2026-03-10T23:59:59</FechaInicio>\r\n    <FechaTermino xsi:nil="true" />\r\n    <Diputado>\r\n      <Id>1096</Id>\r\n      <Nombre>María Candelaria</Nombre>\r\n      <Nombre2 />\r\n      <ApellidoPaterno>Acevedo</ApellidoPaterno>\r\n      <ApellidoMaterno>Sáez</ApellidoMaterno>\r\n      <FechaNacimiento>1958-09-12T00:00:00</FechaNacimiento>\r\n      <RUT />\r\n      <RUTDV />\r\n      <Sexo Valor="0">Femenino</Sexo>\r\n      <Militancias>\r\n        <Militancia>\r\n          <FechaInicio>2022-03-11T00:00:00</FechaInicio>\r\n          <FechaTermino>2026-03-10T23:59:59</FechaTermino>\r\n          <Partido>\r\n            <Id>PC</Id>\r\n            <Nombre>Partido Comunista</Nombre>\r\n            <Alias>PC</Alias>\r\n         

In [5]:
# parse XML using ElementTree, setting the data in a tree structure
deputies_parsed = et.fromstring(deputies_xml)
deputies_parsed

# since in the tree structure the namespace gets inhered fir every children and grandchildren of the root, we have to clean and reduce it
# if we want to work with the tags we need to find a way to work also with the namespace
ns = {"ns": "http://opendata.camara.cl/camaradiputados/v1"}





In [6]:
# we create list to store the inform
deputies_list = []
parties_list = []
militancies_list = []

# loop over each deputy
for dp in deputies_parsed.findall("ns:DiputadoPeriodo", ns):
    # first the information on the term boundaries
    term_start = dp.findtext("ns:FechaInicio", default=None, namespaces=ns)
    term_end = dp.findtext("ns:FechaTermino", default=None, namespaces=ns)
    # now we know that the term_start is really the end of the electoral process

    # now nested inside there's the concrete information on the deputy
    diputado = dp.find("ns:Diputado", ns)
    if diputado is None:
        continue

    diputado_id = diputado.findtext("ns:Id", default=None, namespaces=ns)
    if diputado_id is None:
        continue
    dipid = int(diputado_id)

    # the sex element has text and attribute
    sex_element = diputado.find("ns:Sexo", ns)
    sex_text = sex_element.text if sex_element is not None else None
    sex_value = sex_element.attrib.get("Valor") if sex_element is not None else None
    
    deputies_list.append({
        "dipid": dipid,
        "nombre": clean(diputado.findtext("ns:Nombre", default=None, namespaces=ns)), # some deputies have two names, but they are all in the same tag
        "apellido_paterno": clean(diputado.findtext("ns:ApellidoPaterno", default=None, namespaces=ns)),
        "apellido_materno": clean(diputado.findtext("ns:ApellidoMaterno", default=None, namespaces=ns)),
        "fecha_nacimiento": clean(diputado.findtext("ns:FechaNacimiento", default=None, namespaces=ns)),
        "sex_text": sex_text,
        "sex_value": sex_value
    })

    # now we proceeed with the information on the militancies
    militancias = diputado.find("ns:Militancias", ns)
    if militancias is None:
        continue
    for i in militancias.findall("ns:Militancia", ns):
        i_start = i.findtext("ns:FechaInicio", default=None, namespaces=ns)
        i_end = i.findtext("ns:FechaTermino", default=None, namespaces=ns)

        partido = i.find("ns:Partido", ns)
        if partido is None:
            continue

        party_id = partido.findtext("ns:Id", default=None, namespaces=ns)
        party_name = partido.findtext("ns:Nombre", default=None, namespaces=ns)
        party_alias = partido.findtext("ns:Alias", default=None, namespaces=ns)

        # we save the information on the parties that conform the Congress
        parties_list.append({
            "party_id": party_id,
            "party_name": party_name,
            "party_alias": party_alias
        })

        # finally we save a table linking deputies to their parties
        militancies_list.append({
            "dipid": dipid,
            "party_id": party_id,
            "date_start": i_start,
            "date_end": i_end
        })
    
    dip_df = pd.DataFrame(deputies_list)
    parties_df = pd.DataFrame(parties_list).drop_duplicates(subset=["party_id"])
    militancies_df = pd.DataFrame(militancies_list).drop_duplicates(subset=["dipid", "party_id", "date_start"])

For now we aren't trying to get the deputies making changes, if I want to do it later I have to consider distinguishing between the instances where someone is registerd as changing party and changes to the same one PS -> PS -> PS (e.g. Jaime Naranjo)

## Getting District and Comuna Information from BCN API

Now we'll call a secondary API from BCN to get district and comuna information for active deputies. This supplements the existing data with geographic details.

[This the secondary API with the district, comuna and region information](https://datos.bcn.cl/catalogo/servicio/ServiciosWebHistoriaDeLaLey/ObtenerParlamentariosActivos)

In [7]:
# BCN API endpoint for active parliamentary members
bcn_url = "https://datos.bcn.cl/catalogo/servicio/ServiciosWebHistoriaDeLaLey/ObtenerParlamentariosActivos"

try:
    # Call the BCN API
    bcn_response = requests.get(bcn_url, timeout=30)
    bcn_response.raise_for_status()
    bcn_xml = bcn_response.text
    
    # Save the response for reference
    with open("bcn_parlamentarios_activos.xml", "w", encoding="utf-8") as f:
        f.write(bcn_xml)
    
    print(f"Successfully fetched BCN parliamentary data ({len(bcn_xml)} bytes)")
except requests.RequestException as e:
    print(f"Error fetching BCN API: {e}")
    bcn_xml = None

Successfully fetched BCN parliamentary data (709743 bytes)


In [8]:
# Parse the BCN XML response
districts_list = []

if bcn_xml is not None:
    try:
        bcn_parsed = et.fromstring(bcn_xml)
        
        # BCN API uses a different namespace - we need to find it from the root element
        # The namespace is typically in the format {namespace}tag
        bcn_ns_match = bcn_parsed.tag.split('}')[0].strip('{') if '}' in bcn_parsed.tag else None
        
        if bcn_ns_match:
            bcn_ns = {"bcn": bcn_ns_match}
        else:
            bcn_ns = {}
        
        # Find all parliamentary members in the response
        # The structure may vary, so we'll search for common elements
        parlamentarios = bcn_parsed.findall(".//bcn:Parlamentario", bcn_ns) if bcn_ns else bcn_parsed.findall(".//Parlamentario")
        
        if not parlamentarios:
            # Try without namespace prefix
            parlamentarios = bcn_parsed.iter()
            parlamentarios = [elem for elem in parlamentarios if 'Parlamentario' in str(type(elem))]
        
        print(f"Found {len(parlamentarios)} parliamentary members in BCN API")
        
        for parl in parlamentarios:
            # FILTER: Check if this person is a deputy (Cámara de Diputados), not a senator (Senado)
            camara_elem = parl.find(".//bcn:Camara", bcn_ns) if bcn_ns else parl.find(".//Camara")
            if camara_elem is not None:
                camara_descripcion = camara_elem.findtext(".//bcn:Descripcion", default=None, namespaces=bcn_ns) if bcn_ns else camara_elem.findtext(".//Descripcion")
                if camara_descripcion is None:
                    # Try direct text if Descripcion is a direct child
                    camara_descripcion_elem = camara_elem.find(".//bcn:Descripcion", bcn_ns) if bcn_ns else camara_elem.find(".//Descripcion")
                    camara_descripcion = camara_descripcion_elem.text if camara_descripcion_elem is not None else None
                
                # Only process deputies, skip senators
                if camara_descripcion != "Cámara de Diputados":
                    continue
            
            # Extract IdEnCamaraDeOrigen which maps to our dipid
            id_elem = parl.find(".//bcn:IdEnCamaraDeOrigen", bcn_ns) if bcn_ns else parl.find(".//IdEnCamaraDeOrigen")
            if id_elem is None or id_elem.text is None:
                continue
            
            try:
                dipid = int(id_elem.text)
            except (ValueError, TypeError):
                continue
            
            # Extract geographic representation information
            rep_geo = parl.find(".//bcn:RepresentacionGeografica", bcn_ns) if bcn_ns else parl.find(".//RepresentacionGeografica")
            
            district_name = None
            district_number = None
            region = None
            comunas_text = []
            
            if rep_geo is not None:
                # Extract district name from DivisionPoliticoAdministrativa
                div_politico = rep_geo.find(".//bcn:DivisionPoliticoAdministrativa", bcn_ns) if bcn_ns else rep_geo.find(".//DivisionPoliticoAdministrativa")
                if div_politico is not None:
                    # The descripcion tag is lowercase here
                    desc_elem = div_politico.find("bcn:descripcion", bcn_ns) if bcn_ns else div_politico.find("descripcion")
                    if desc_elem is None:
                        desc_elem = div_politico.find("bcn:Descripcion", bcn_ns) if bcn_ns else div_politico.find("Descripcion")
                    district_name = desc_elem.text if desc_elem is not None else None
                    
                    # Try to extract district number (often in the descripcion format like "Circunscripción V")
                    # For now using descripcion as is
                
                # Extract region name
                region_elem = rep_geo.find(".//bcn:Region", bcn_ns) if bcn_ns else rep_geo.find(".//Region")
                if region_elem is not None:
                    region_desc = region_elem.find(".//bcn:Descripcion", bcn_ns) if bcn_ns else region_elem.find(".//Descripcion")
                    region = region_desc.text if region_desc is not None else None
                
                # Extract comunas
                comunas_parent = rep_geo.find(".//bcn:Comunas", bcn_ns) if bcn_ns else rep_geo.find(".//Comunas")
                if comunas_parent is not None:
                    comuna_elems = comunas_parent.findall(".//bcn:Comuna", bcn_ns) if bcn_ns else comunas_parent.findall(".//Comuna")
                    for com in comuna_elems:
                        desc = com.find(".//bcn:Descripcion", bcn_ns) if bcn_ns else com.find(".//Descripcion")
                        if desc is not None and desc.text:
                            comunas_text.append(desc.text)
            
            districts_list.append({
                "dipid": dipid,
                "district_number": district_number,
                "district_name": district_name,
                "comunas": ", ".join(comunas_text) if comunas_text else None,
                "region": region,
                "data_source": "BCN_API"
            })
        
        print(f"Successfully parsed district data for {len(districts_list)} deputies (senators filtered out)")
        
    except Exception as e:
        print(f"Error parsing BCN XML: {e}")
        import traceback
        traceback.print_exc()
        print("Continuing without district enrichment...")
else:
    print("No BCN data to parse")

# Create DataFrame for validation
if districts_list:
    districts_df = pd.DataFrame(districts_list)
    print(f"\nDistricts DataFrame shape: {districts_df.shape}")
    print(f"Sample of district data:")
    print(districts_df.head())
else:
    districts_df = pd.DataFrame(columns=["dipid", "district_number", "district_name", "comunas", "region", "data_source"])

Found 205 parliamentary members in BCN API
Successfully parsed district data for 155 deputies (senators filtered out)

Districts DataFrame shape: (155, 6)
Sample of district data:
   dipid district_number   district_name  \
0   1165            None   Distrito N° 8   
1   1179            None   Distrito N° 8   
2   1160            None  Distrito N° 10   
3   1013            None  Distrito N° 26   
4   1076            None  Distrito N° 17   

                                             comunas  \
0  Cerrillos, Colina, Estación Central, Lampa, Ma...   
1  Cerrillos, Colina, Estación Central, Lampa, Ma...   
2  La Granja, Macul, Ñuñoa, Providencia, San Joaq...   
3  Ancud, Calbuco, Castro, Chaitén, Chonchi, Coch...   
4  Constitución, Curepto, Curicó, Empedrado, Hual...   

                             region data_source  
0  Región Metropolitana De Santiago     BCN_API  
1  Región Metropolitana De Santiago     BCN_API  
2  Región Metropolitana De Santiago     BCN_API  
3               Re

In [9]:
# Data validation: Compare deputies from both APIs
if not districts_df.empty and 'dipid' in dip_df.columns:
    deputies_with_districts = set(districts_df['dipid'].unique())
    all_deputies = set(dip_df['dipid'].unique())
    
    matched = deputies_with_districts.intersection(all_deputies)
    missing_districts = all_deputies - deputies_with_districts
    extra_districts = deputies_with_districts - all_deputies
    
    print(f"\n{'='*60}")
    print(f"DATA VALIDATION SUMMARY")
    print(f"{'='*60}")
    print(f"Total deputies from primary API: {len(all_deputies)}")
    print(f"Deputies with BCN district data: {len(deputies_with_districts)}")
    print(f"Successfully matched: {len(matched)} ({len(matched)/len(all_deputies)*100:.1f}%)")
    
    if missing_districts:
        print(f"\nDeputies missing district data: {len(missing_districts)}")
        missing_names = dip_df[dip_df['dipid'].isin(missing_districts)][['dipid', 'nombre', 'apellido_paterno']].head(10)
        print(missing_names.to_string(index=False))
        if len(missing_districts) > 10:
            print(f"  ... and {len(missing_districts) - 10} more")
    
    if extra_districts:
        print(f"\nDeputies in BCN API but not in primary API: {len(extra_districts)}")
        print(f"  IDs: {sorted(list(extra_districts))[:10]}")
    
    print(f"{'='*60}\n")


DATA VALIDATION SUMMARY
Total deputies from primary API: 157
Deputies with BCN district data: 155
Successfully matched: 155 (98.7%)

Deputies missing district data: 2
 dipid   nombre apellido_paterno
  1115 Mercedes           Bulnes
  1124    Tomás    De Rementería



## SQL tables

now we try to get the lists into the SQL tables

In [16]:
con = sqlite3.connect("chilean_polispeech.db")
cur = con.cursor()

# Execute CREATE TABLE statements separately (SQLite doesn't support multiple statements in a single execute)
cur.execute("""
CREATE TABLE IF NOT EXISTS deputies (
    dipid INT PRIMARY KEY, 
    name TEXT,
    lastname_1 TEXT,
    lastname_2 TEXT,
    birth_date TEXT,            
    sex_value TEXT,            
    sex_text TEXT            
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS parties (
    party_id INT PRIMARY KEY,        
    party_name TEXT,
    party_alias TEXT
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS militancies (
    dipid INT,
    party_id INT,
    date_start TEXT,
    date_end TEXT,
    PRIMARY KEY (dipid, party_id, date_start),
    FOREIGN KEY (dipid) REFERENCES deputies (dipid),
    FOREIGN KEY (party_id) REFERENCES parties (party_id)
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS districts (
    dipid INT PRIMARY KEY,
    district_number TEXT,
    district_name TEXT,
    comunas TEXT,
    region TEXT,
    data_source TEXT,
    FOREIGN KEY (dipid) REFERENCES deputies (dipid)
);
""")

con.commit()

# Rename DataFrame columns to match SQL table schema
dip_df_renamed = dip_df.rename(columns={
    "nombre": "name",
    "apellido_paterno": "lastname_1",
    "apellido_materno": "lastname_2",
    "fecha_nacimiento": "birth_date"
})

# Insert deputies, parties, and militancies data
try:
    dip_df_renamed.to_sql("deputies", con, if_exists="append", index=False)
    print(f"✓ Inserted {len(dip_df_renamed)} deputy records into database")
except Exception as e:
    print(f"✗ Error inserting deputies: {e}")

try:
    parties_df.to_sql("parties", con, if_exists="append", index=False)
    print(f"✓ Inserted {len(parties_df)} party records into database")
except Exception as e:
    print(f"✗ Error inserting parties: {e}")

try:
    militancies_df.to_sql("militancies", con, if_exists="append", index=False)
    print(f"✓ Inserted {len(militancies_df)} militancy records into database")
except Exception as e:
    print(f"✗ Error inserting militancies: {e}")

# Insert districts data if available
if not districts_df.empty:
    try:
        districts_df.to_sql("districts", con, if_exists="append", index=False)
        print(f"✓ Inserted {len(districts_df)} district records into database")
    except Exception as e:
        print(f"✗ Error inserting districts: {e}")
else:
    print("⚠ No district data to insert")

con.close()
print("\n✓ Database update completed!")

✗ Error inserting deputies: UNIQUE constraint failed: deputies.dipid
✗ Error inserting parties: UNIQUE constraint failed: parties.party_id
✗ Error inserting militancies: UNIQUE constraint failed: militancies.dipid, militancies.party_id, militancies.date_start
✗ Error inserting districts: UNIQUE constraint failed: districts.dipid

✓ Database update completed!


In [11]:
type(militancies_list)

list

In [12]:
districts_df

,dipid,district_number,district_name,comunas,region,data_source
0,1165,None,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago,BCN_API
1,1179,None,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago,BCN_API
2,1160,None,Distrito N° 10,"La Granja, Macul, Ñuñoa, Providencia, San Joaq...",Región Metropolitana De Santiago,BCN_API
3,1013,None,Distrito N° 26,"Ancud, Calbuco, Castro, Chaitén, Chonchi, Coch...",Región De Los Lagos,BCN_API
4,1076,None,Distrito N° 17,"Constitución, Curepto, Curicó, Empedrado, Hual...",Región Del Maule,BCN_API
...,...,...,...,...,...,...
150,1158,None,Distrito N° 5,"Andacollo, Canela, Combarbalá, Coquimbo, Illap...",Región De Coquimbo,BCN_API
151,1125,None,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago,BCN_API
152,991,None,Distrito N° 1,"Arica, Camarones, General Lagos, Putre",Región De Arica Y Parinacota,BCN_API
153,1059,None,Distrito N° 12,"La Florida, La Pintana, Pirque, Puente Alto, S...",Región Metropolitana De Santiago,BCN_API


In [13]:
deputies_parsed.get(".//ns:diputado", ns)

{'ns': 'http://opendata.camara.cl/camaradiputados/v1'}